# Tata Technologies Ltd. - TechPulse FY-26: Applied AI & ML
## Lab Statement 5: Predictive Maintenance from Sensor Logs

**Curriculum Context:** Unit 1 (Automotive Use Cases: Predictive Maintenance) & Unit 4 (Evaluation Metrics: Precision, Recall, F1-Score, ROC-AUC)  
**Track:** AI & ML | **Level:** Intermediate  
**Dataset:** Industrial Automotive Sensor Telemetry (`sensor_maintenance_data.csv`)  
**Domain Focus:** Equipment Degradation Detection, Failure Mode Classification & Condition-Based Maintenance  

---

### 🎯 Learning Objectives
1. Analyze multi-channel sensor telemetry streams (temperature, rotational velocity, torque, tool wear).
2. Address class imbalance through stratified sampling and cost-sensitive balanced class weighting.
3. Implement and benchmark binary/multi-class classifiers: Logistic Regression, Decision Trees, Random Forests, SVC, and Gradient Boosting.
4. Compute and interpret industrial evaluation metrics: Precision, Recall, F1-Score, and ROC-AUC.
5. Evaluate confusion matrices and identify dominant sensor degradation predictors.

---

### Step 0: Imports & Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, roc_curve, precision_score, recall_score, f1_score, accuracy_score

plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['figure.dpi'] = 120
print("Environment initialized!")

### Step 1: Ingesting Sensor Logs & Failure Modes

In [ ]:
df = pd.read_csv('sensor_maintenance_data.csv')
print(f"Total Sensor Records: {df.shape[0]} | Columns: {df.shape[1]}")
print(f"Normal: {(df['failure']==0).sum()} | Failure: {(df['failure']==1).sum()} ({(df['failure'].mean())*100:.1f}%)")
df.head()

### Step 2: Exploratory Visualizations & Failure Distributions

In [ ]:
from IPython.display import Image
Image('plots/01_sensor_distributions_and_failures.png')

### Step 3: Train-Test Split (Stratified) & Model Benchmarking

In [ ]:
feature_cols = ['air_temperature_k', 'process_temperature_k', 'rotational_speed_rpm', 'torque_nm', 'tool_wear_min']
X = df[feature_cols]
y = df['failure']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42, stratify=y)

models = {
    'Logistic Regression': Pipeline([('scaler', StandardScaler()), ('clf', LogisticRegression(class_weight='balanced', random_state=42))]),
    'Decision Tree Clf': Pipeline([('scaler', StandardScaler()), ('clf', DecisionTreeClassifier(max_depth=5, class_weight='balanced', random_state=42))]),
    'Random Forest Clf': Pipeline([('scaler', StandardScaler()), ('clf', RandomForestClassifier(n_estimators=100, max_depth=8, class_weight='balanced', random_state=42, n_jobs=1))]),
    'Support Vector Clf': Pipeline([('scaler', StandardScaler()), ('clf', SVC(kernel='rbf', class_weight='balanced', random_state=42))]),
    'Gradient Boosting Clf': Pipeline([('scaler', StandardScaler()), ('clf', GradientBoostingClassifier(n_estimators=100, learning_rate=0.08, max_depth=4, random_state=42))])
}

results = []
for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred)
    rec = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    results.append({'Classifier': name, 'Accuracy': round(acc, 4), 'Precision': round(prec, 4), 'Recall': round(rec, 4), 'F1-Score': round(f1, 4)})

pd.DataFrame(results)

### Step 4: Diagnostic Confusion Matrices & ROC-AUC Curves

In [ ]:
Image('plots/02_confusion_matrix_and_roc_curves.png')

In [ ]:
Image('plots/03_sensor_feature_importance.png')

### Step 5: Real-Time Diagnostic Sensor Test Function

In [ ]:
best_clf = models['Random Forest Clf']

def diagnose_sensor_readings(air_temp_k, process_temp_k, rpm, torque_nm, tool_wear_min):
    reading = pd.DataFrame([{
        'air_temperature_k': air_temp_k,
        'process_temperature_k': process_temp_k,
        'rotational_speed_rpm': rpm,
        'torque_nm': torque_nm,
        'tool_wear_min': tool_wear_min
    }])
    pred = best_clf.predict(reading)[0]
    status = "⚠️ IMPENDING COMPONENT FAILURE DETECTED" if pred == 1 else "✅ NORMAL COMPONENT OPERATION"
    print(f"Telemetry Diagnostic Status: {status}")

# Normal Condition Test:
diagnose_sensor_readings(air_temp_k=298.5, process_temp_k=308.2, rpm=1500, torque_nm=38.0, tool_wear_min=45.0)

# Overstrain / High Torque Wear Test:
diagnose_sensor_readings(air_temp_k=303.0, process_temp_k=314.5, rpm=1250, torque_nm=72.0, tool_wear_min=220.0)